In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RecoScale-ALS-Train") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.memory.fraction", "0.7") \
    .config("spark.memory.storageFraction", "0.2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.local.dir", "/mnt/d/spark_tmp") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.network.timeout", "300s") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()
print('Spark Started...')

Spark Started...


In [3]:
train = spark.read.parquet("/mnt/d/Projects/recoscale/data/train.parquet")
test = spark.read.parquet("/mnt/d/Projects/recoscale/data/test.parquet")

In [5]:
from pyspark.sql import functions as F
gt  = test.groupBy("user_idx").agg(F.collect_set("item_idx").alias("gt_items")).cache()

users = gt.sample(fraction=0.25, seed=42).select("user_idx").cache()

In [8]:
gt.printSchema()

root
 |-- user_idx: integer (nullable = true)
 |-- gt_items: array (nullable = false)
 |    |-- element: integer (containsNull = false)



In [7]:
from pyspark.ml.recommendation import ALSModel

model = ALSModel.load("/mnt/d/Projects/recoscale/models/als_(10,0.01)_model")

small_users = users.limit(5)

k = 10
recs_small = model.recommendForUserSubset(small_users, k) \
    .withColumn("rec_items", F.col("recommendations.item_idx"))

recs_small.printSchema()

root
 |-- user_idx: integer (nullable = false)
 |-- recommendations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_idx: integer (nullable = true)
 |    |    |-- rating: float (nullable = true)
 |-- rec_items: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [9]:
gt_sampled = gt.join(users, "user_idx")
df = recs_small.join(gt_sampled, "user_idx")
df.show(5)

26/04/13 05:37:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+--------+--------------------+--------------------+----------+
|user_idx|     recommendations|           rec_items|  gt_items|
+--------+--------------------+--------------------+----------+
|    7833|[{10831581, 9.919...|[10831581, 699010...|[16479073]|
+--------+--------------------+--------------------+----------+



In [4]:
train_items = train.select("item_idx").distinct()
test_items = test.select("item_idx").distinct()

In [5]:
print("Train items:", train.select("item_idx").distinct().count())
print("Test items:", test.select("item_idx").distinct().count())
print("Overlap:", train_items.intersect(test_items).count())

Train items: 18362145


Test items: 5549707


Overlap: 4520015


In [ ]:
print(f"Number of rows in training data before sampling: {train.count()}")
print(f"Number of rows in testing data before sampling: {test.count()}")

Number of rows in training data before sampling: 221334514


Number of rows in testing data before sampling: 30697688


In [11]:
print(f"Number of distinct rows in training data before sampling: {train_df.select('user_idx').distinct().count()}")
print(f"Number of distinct rows in testing data before sampling: {test_df.select('user_idx').distinct().count()}")

Number of distinct rows in training data before sampling: 44257499


Number of distinct rows in testing data before sampling: 30697688


In [12]:
user_sampling = train_df.select('user_idx').distinct().sample(fraction= 0.1, seed= 42)

In [13]:
train_df = train_df.join(user_sampling, on= 'user_idx').repartition(200, 'user_idx')
test_df = test_df.join(user_sampling, on= 'user_idx').repartition(200, 'user_idx')

In [6]:
print(f"Number of rows in training data after sampling: {train_df.count()}")
print(f"Number of rows in testing data after sampling: {test_df.count()}")

Number of rows in training data after sampling: 22133905


Number of rows in testing data after sampling: 3071844


In [7]:
distinct_test = test_df.select('user_idx').distinct()
print(f"Number of distict rows in test data after sampling: {distinct_test.count()}")

Number of distict rows in test data after sampling: 3071844


In [8]:
distinct_train = train_df.select('user_idx').distinct()
print(f"Number of distict rows in train data after sampling: {distinct_train.count()}")

26/04/12 07:30:23 WARN TaskMemoryManager: Failed to allocate a page (1073741824 bytes), try again.


Number of distict rows in train data after sampling: 4425559
